#                                                 **FINAL MODEL+EDA+OBSERVATIONS+ENSEMBLE**

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pd.options.display.max_columns=100
small=0.00001 #to prevent division by zero

In [ ]:
train_df=pd.read_csv("../input/beyond-analysis/train.csv")
test_df=pd.read_csv("../input/beyond-analysis/test.csv")

# **Lets check shape of data**

In [ ]:
print("shape of train:",train_df.shape)
print("shape of test:",test_df.shape)

# **Check NULL values in our data**

In [ ]:
# check if there are null values or not
train_df.isnull().sum()

# **Check NULL values in our data**

In [ ]:
# check if there are null values or not
test_df.isnull().sum()

# **Check data types of values values in our data**

In [ ]:
train_df.info()

# **Lets gain information about the columns to be predicted**

In [ ]:
print("the maximum value in the column Y1 is:",train_df['Y1'].max())
print("the minimum value in the column Y1 is:",train_df['Y1'].min())
print("the mean value in the column Y1 is:",train_df['Y1'].mean())
print("the standard deviation value in the column Y1 is:",train_df['Y1'].std())
print('-'*100)
print("the maximum value in the column Y1 is:",train_df['Y2'].max())
print("the minimum value in the column Y1 is:",train_df['Y2'].min())
print("the mean value in the column Y1 is:",train_df['Y2'].mean())
print("the standard deviation value in the column Y1 is:",train_df['Y2'].std())

In [ ]:
data=train_df.groupby(['UNIQUE_IDENTIFIER'])['Y1','Y2'].mean()

In [ ]:
print('-'*100)
print("the maximum value in the column Y1 is:",data['Y1'].max())
print("the minimum value in the column Y1 is:",data['Y1'].min())
print("the mean value in the column Y1 is:",data['Y1'].mean())
print("the standard deviation value in the column Y1 is:",data['Y1'].std())
print('-'*100)
print("the maximum value in the column Y1 is:",data['Y2'].max())
print("the minimum value in the column Y1 is:",data['Y2'].min())
print("the mean value in the column Y1 is:",data['Y2'].mean())
print("the standard deviation value in the column Y1 is:",data['Y2'].std())
print('-'*100)

# **OBSERVATIONS**
* there is a huge standard deviation in our Y2 column so with reference to the evaluation metric of the competition it is more important to reduce the RMSE of Y2 metric than Y1 metric
* so we recieved a significant improvement in the variance of both columns if we group the data on the basis of unique identifier.
* this will lead to loss of information therefore the main crux we have to do is how can we retain a lot of information for my model to perform better we can use min, max, mean, std values, appart from that we can use a lot amount of feature engineering which is done below

**OBSERVATIONS** 
* **Y1 and Y2 are labels to be predicted** 
* **This is a r problem**
* **there are no missing values in test and train set**

In [ ]:
#display(train_df.target.describe())
f, ax = plt.subplots(nrows=2, ncols=3, figsize=(18, 4))
sns.distplot(train_df.Y1, ax=ax[0,0])
sns.boxplot(train_df.Y1, ax=ax[0,1])
stats.probplot(train_df['Y1'], plot=ax[0,2])

sns.distplot(train_df.Y2, ax=ax[1,0])
sns.boxplot(train_df.Y2, ax=ax[1,1])
stats.probplot(train_df['Y2'], plot=ax[1,2])

plt.tight_layout()
plt.show()

# **OBSERVATIONS:**
* There is positively skewed skewness in the metrices to be predicted,the reason is for a particular id there are many sparse values so this also indicates that we should try grouping our data to remove the skewness.
* for metrix Y2 prob plot showed logistic distribution between observed and theoritical values
* There are huge  outliers in our data (for ex there is a value above 800 that should be removed).

In [ ]:
features = [feature for feature in train_df.columns if feature not in ['UNIQUE_IDENTIFIER', 'Y1','Y2','CATEGORY_1','CATEGORY_2']]

fig = plt.figure(figsize=(12, 12), facecolor='#f6f6f6')
gs = fig.add_gridspec(5, 4)
gs.update(wspace=0.1, hspace=0.4)

background_color = "#f6f6f6"

run_no = 0
for row in range(0, 5):
    for col in range(0, 4):
        locals()["ax"+str(run_no)] = fig.add_subplot(gs[row, col])
        locals()["ax"+str(run_no)].set_facecolor(background_color)
        locals()["ax"+str(run_no)].tick_params(axis='y', left=False)
        locals()["ax"+str(run_no)].get_yaxis().set_visible(False)
        for s in ["top","right","left"]:
            locals()["ax"+str(run_no)].spines[s].set_visible(False)
        run_no += 1

run_no = 0
for feature in features:
        sns.kdeplot(train_df[feature] ,ax=locals()["ax"+str(run_no)], color='#ffd514', shade=True, linewidth=1.5, alpha=0.9, zorder=3, legend=False)
        locals()["ax"+str(run_no)].grid(which='major', axis='x', zorder=0, color='gray', linestyle=':', dashes=(1,5))
        locals()["ax"+str(run_no)].set_xlabel(feature)
        run_no += 1


# **OBSERVATIONS:**
* all the continuous features are also left skewed they reason is obvioiusly because of huge amount of 0 values present in the data.
* we can try to remove this skewness by converting the scaling the data with the help of quantile transformer.(in processing section).
* also since it is required that we have to predict one value for each unique identifier we can also group the data which significantly reduces the 0 values.

In [ ]:
f, ax = plt.subplots(nrows=1, ncols=1, figsize=(20, 20))
ax.set_title("Correlation Matrix", fontsize=16)
sns.heatmap(train_df[train_df.columns[train_df.columns != 'UNIQUE_IDENTIFIER']].corr(), vmin=-1, vmax=1, cmap='coolwarm', annot=True)

for tick in ax.xaxis.get_major_ticks():
    tick.label.set_fontsize(14) 
    tick.label.set_rotation(90) 
for tick in ax.yaxis.get_major_ticks():
    tick.label.set_fontsize(14)
    tick.label.set_rotation(0) 
plt.show()

# **CORRELATION OBSERVATIONS**
* there is sinificant correlation between Y2 and sequence size and status check.
* There seems to be huge correlation between **ENTRY ,REVENUE AND WINNINGS_1**
* Y2 seems to be fairly correlated with **SEQUENCE_NO,STATUS_CHECK** 
* there seems to be fair correlation between variouse columns hence combined polynomial featues can be build 


# **Categorical Features** 

In [ ]:
print("unique values in category 1 are:(train set) ",train_df['CATEGORY_1'].unique())
print("unique values in category 1 are(test set): ",test_df['CATEGORY_1'].unique())

In [ ]:
print("unique values in category 2 are:(train set) ",train_df['CATEGORY_2'].unique())
print("unique values in category 2 are(test set): ",test_df['CATEGORY_2'].unique())

# **OBSERVATIONS**
* Not all the values in train and test data are same.

* We can perform simple label encoding on the categorical features.

# **FEATURE ENGINEERING/EXTRACTION**

In [ ]:
# corr_featu=["DEPOSIT", "ENTRY", "REVENUE", "WINNINGS_1"]

In [ ]:
#function to find the difference in the series
def differ(Series):
    return Series.diff()

# **Description of columns**
* pct is simply the difference between deposit and entry(important).
* fract is the ratio fun contest wins and the fun contests participated.
* fract1 is the ratio  contest wins and the fun contests participated(very important).
* fract2 is the ratio  contest wins and the fun contests participated.
* paid is difference of revenue and discount
* fract4 is difference of  entry and winnings1(total chips wins)

In [ ]:
train_df[train_df['DISCOUNT']<=train_df['ENTRY']]

In [ ]:
# difference in the cards to be purchased an actual cards purchased
train_df['pct']=(train_df['DEPOSIT']-train_df['ENTRY'])
test_df['pct']=(test_df['DEPOSIT']-test_df['ENTRY']) 
#
train_df['fract']=train_df['PRACTICE_WINNINGS_NUMBER']/(train_df['PRACTICE_ENTRY_NUMBER']+0.00001)
test_df['fract']=test_df['PRACTICE_WINNINGS_NUMBER']/(test_df['PRACTICE_ENTRY_NUMBER']+0.00001)
# ENTRY_NUMBER - Contests participated
# WINNINGS_NUMBER - Contests won
train_df['fract1']=train_df['WINNINGS_NUMBER']/(train_df['ENTRY_NUMBER']+0.00001)
test_df['fract1']=test_df['WINNINGS_NUMBER']/(test_df['ENTRY_NUMBER']+0.00001)

train_df['fract2']=train_df['PRACTICE_WINNINGS']/(train_df['PRACTICE_ENTRY']+0.00001)
test_df['fract2']=test_df['PRACTICE_WINNINGS']/(test_df['PRACTICE_ENTRY']+0.00001)

train_df['fract3']=(train_df['WITHDRAW']-train_df['DEPOSIT'])/(train_df['WITHDRAW']+train_df['DEPOSIT']+0.00001)
test_df['fract3']=(test_df['WITHDRAW']-test_df['DEPOSIT'])/(test_df['WITHDRAW']+test_df['DEPOSIT']+0.000001)

train_df['paid']=train_df['REVENUE']-train_df['DISCOUNT']
test_df['paid']=test_df['REVENUE']-test_df['DISCOUNT']

train_df['fract4']=train_df['ENTRY']-train_df['WINNINGS_1']
test_df['fract4']=test_df['ENTRY']-test_df['WINNINGS_1']

train_df['fract5']=train_df['REVENUE']/(train_df['ENTRY']+0.01)
test_df['fract5']=test_df['REVENUE']/(test_df['ENTRY']+0.01)

train_df['fract6']=train_df['DISCOUNT']/(train_df['ENTRY']+0.01)
test_df['fract6']=test_df['DISCOUNT']/(test_df['ENTRY']+0.01)

In [ ]:
features = [feature for feature in train_df.columns if feature not in ['UNIQUE_IDENTIFIER','SEQUENCE_NO', 'Y1','Y2','CATEGORY_1','CATEGORY_2']]
for i in features:
    train_df[i+'_diff']=differ(train_df[i])
    test_df[i+'_diff']=differ(test_df[i])

for i in features:
    train_df[i+'_diff'][train_df['SEQUENCE_NO']==1]=0.00
    test_df[i+'_diff'][test_df['SEQUENCE_NO']==1]=0.00

In [ ]:
train_df[train_df['SEQUENCE_NO']==1]

# **PREPROCESSING UNGROUPED DATA**

In [ ]:
import string
cat_columns=['CATEGORY_1', 'CATEGORY_2']
from sklearn.preprocessing import LabelEncoder
lb=LabelEncoder()
lb.fit(list(string.ascii_uppercase))
for x in cat_columns:
    train_df[x]=lb.transform(train_df[x])
    test_df[x]=lb.transform(test_df[x])

# **GROUPING**
* Grouping the data on the basis of unique identifer, this is surely going to reduce the 0 values in the data and possibly introduce less skewed variables it helped me in reducing standard deviation.

In [ ]:
def percentile(n):
    def percentile_(x):
        return np.percentile(x, n)
    percentile_.__name__ = 'percentile_%s' % n
    return percentile_
i=[np.mean,np.max,np.min,np.sum,np.std,np.median]
# i=[np.mean,np.max,np.min,np.sum,np.std,np.median]
fe_dict = {
        'SEQUENCE_NO':[np.size],
        'STATUS_CHECK':[np.mean],
        'CATEGORY_1':[np.mean],
        'CATEGORY_2':[np.mean],
        'ACTIVE_YN':[np.mean,np.sum],
        'ENTRY':i,
        'REVENUE':i,
        'WINNINGS_1':i,
        'WINNINGS_2':i,
        'DISCOUNT':i,
        'DEPOSIT':i,
        'DEPOSIT_NUMBER':i,
        'DEPOSIT_2':i,
        'WITHDRAW':i,
        'WITHDRAW_NUMBER':i,
        'DEPOSIT_TRAILS':i,
        'ENTRY_NUMBER':i,
        'WINNINGS_NUMBER':i,
        'PRACTICE_ENTRY':i,
        'PRACTICE_WINNINGS':i,
        'PRACTICE_ENTRY_NUMBER':i,
        'PRACTICE_WINNINGS_NUMBER':i,
        'ACTIVE_YN_diff':[np.mean,np.sum],
        'ENTRY_diff':i,
        'REVENUE_diff':i,
        'WINNINGS_1_diff':i,
         'WINNINGS_2_diff':i,
        'DISCOUNT_diff':i,
        'DEPOSIT_diff':i,
        'DEPOSIT_NUMBER_diff':i,
        'DEPOSIT_2_diff':i,
        'WITHDRAW_diff':i,
        'WITHDRAW_NUMBER_diff':i,
        'DEPOSIT_TRAILS_diff':i,
        'ENTRY_NUMBER_diff':i,
        'WINNINGS_NUMBER_diff':i,
        'PRACTICE_ENTRY_diff':i,
        'PRACTICE_WINNINGS_diff':i,
        'PRACTICE_ENTRY_NUMBER_diff':i,
        'PRACTICE_WINNINGS_NUMBER_diff':i,
        'pct':i,
        'fract':i,
        'fract1':i,
        'fract2':i,
        'fract3':i,
        'fract4':i,
        'fract5':i,
        'fract6':i,
        'paid':i,
        'Y1':[np.mean],
        'Y2':[np.mean],
         }
train=train_df.groupby(['UNIQUE_IDENTIFIER']).agg(fe_dict).reset_index()
train.columns = ['_'.join(col) for col in train.columns]

In [ ]:
f, ax = plt.subplots(nrows=1, ncols=1, figsize=(6,6))
#f.suptitle('Distribution of Features', fontsize=16)
sns.distplot(train['DEPOSIT_sum'])
plt.tight_layout()
plt.show()

In [ ]:
fe_dict = {
        'SEQUENCE_NO':[np.size],
        'STATUS_CHECK':[np.mean],
        'CATEGORY_1':[np.mean],
        'CATEGORY_2':[np.mean],
        'ACTIVE_YN':[np.mean,np.sum],
        'ENTRY':i,
        'REVENUE':i,
        'WINNINGS_1':i,
        'WINNINGS_2':i,
        'DISCOUNT':i,
        'DEPOSIT':i,
        'DEPOSIT_NUMBER':i,
        'DEPOSIT_2':i,
        'WITHDRAW':i,
        'WITHDRAW_NUMBER':i,
        'DEPOSIT_TRAILS':i,
        'ENTRY_NUMBER':i,
        'WINNINGS_NUMBER':i,
        'PRACTICE_ENTRY':i,
        'PRACTICE_WINNINGS':i,
        'PRACTICE_ENTRY_NUMBER':i,
        'PRACTICE_WINNINGS_NUMBER':i,
        'ACTIVE_YN_diff':[np.mean,np.sum],
        'ENTRY_diff':i,
        'REVENUE_diff':i,
        'WINNINGS_1_diff':i,
         'WINNINGS_2_diff':i,
        'DISCOUNT_diff':i,
        'DEPOSIT_diff':i,
        'DEPOSIT_NUMBER_diff':i,
        'DEPOSIT_2_diff':i,
        'WITHDRAW_diff':i,
        'WITHDRAW_NUMBER_diff':i,
        'DEPOSIT_TRAILS_diff':i,
        'ENTRY_NUMBER_diff':i,
        'WINNINGS_NUMBER_diff':i,
        'PRACTICE_ENTRY_diff':i,
        'PRACTICE_WINNINGS_diff':i,
        'PRACTICE_ENTRY_NUMBER_diff':i,
        'PRACTICE_WINNINGS_NUMBER_diff':i,
        'pct':i,
        'fract':i,
        'fract1':i,
        'fract2':i,
         'fract3':i,
        'fract4':i,
        'fract5':i,
        'fract6':i,
        'paid':i,
         }
test=test_df.groupby(['UNIQUE_IDENTIFIER']).agg(fe_dict).reset_index()
test.columns = ['_'.join(col) for col in test.columns]

In [ ]:
test

In [ ]:
# STATUS_CHECK_mean SEQUENCE_NO_size
train['fract1_']=train['WINNINGS_NUMBER_sum']/(train['ENTRY_NUMBER_sum']+0.00001)
test['fract1_']=test['WINNINGS_NUMBER_sum']/(test['ENTRY_NUMBER_sum']+0.00001)

In [ ]:
train=train.fillna(0.0)
test=test.fillna(0.0)

In [ ]:
features=['Y1_mean','Y2_mean']

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
kmeans = KMeans(n_clusters=10, random_state=2021)
kmeans.fit(train[features])

pca = PCA(n_components=2)
stocks_2d = pca.fit_transform(train[features])

fig, ax = plt.subplots(figsize=(32, 10))
ax.scatter(stocks_2d[:, 0], stocks_2d[:, 1], s=200, c=kmeans.labels_, cmap='RdBu')
# for idx, stock_id in enumerate(train['UNIQUE_IDENTIFIER_'].values):
#     ax.annotate(stock_id, (stocks_2d[idx, 0], stocks_2d[idx, 1]), fontsize=20)
    
ax.tick_params(axis='x', labelsize=20, pad=10)
ax.tick_params(axis='y', labelsize=20, pad=10)
ax.set_title('ID Clusters', size=25, pad=20)
plt.show()

# OBSERVATIONS:
*  based on y1 and y2 there are outliers in our data.
*  I had an idea to produce startified kfold based on clusters i performed it but it didn't worked well so i didn't proceed with it

In [ ]:
print("the maximum value in the column Y1 is:",train['Y1_mean'].max())
print("the minimum value in the column Y1 is:",train['Y1_mean'].min())
print("the mean value in the column Y1 is:",train['Y1_mean'].mean())
print("the standard deviation value in the column Y1 is:",train['Y1_mean'].std())
print('-'*100)
print("the maximum value in the column Y1 is:",train['Y2_mean'].max())
print("the minimum value in the column Y1 is:",train['Y2_mean'].min())
print("the mean value in the column Y1 is:",train['Y2_mean'].mean())
print("the standard deviation value in the column Y1 is:",train['Y2_mean'].std())


# **Scaling Data**

* **QuantileTransformer**

Transform features using quantiles information.

This method transforms the features to follow a uniform or a normal distribution. Therefore, for a given feature, this transformation tends to spread out the most frequent values. It also reduces the impact of (marginal) outliers: this is therefore a robust preprocessing scheme.

The transformation is applied on each feature independently. First an estimate of the cumulative distribution function of a feature is used to map the original values to a uniform distribution. The obtained values are then mapped to the desired output distribution using the associated quantile function. Features values of new/unseen data that fall below or above the fitted range will be mapped to the bounds of the output distribution. Note that this transform is non-linear. It may distort linear correlations between variables measured at the same scale but renders variables measured at different scales more directly comparable.


In [ ]:
features = [col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_"}]

In [ ]:
# Train
from sklearn.preprocessing import QuantileTransformer
train_model_sc = train.copy()
train_label = train_model_sc[["Y1_mean","Y2_mean"]]
train_model_sc = train_model_sc[features]
columns = train_model_sc.columns
# Test
test_model_sc = test.copy()
test_model_sc = test_model_sc[features]
#Scaler
qt = QuantileTransformer(n_quantiles=1000, random_state=0,output_distribution = "normal")
# Scaling
train[features] = qt.fit_transform(train_model_sc)
test[features]  = qt.transform(test_model_sc)

train=train.fillna(0.0)
test=test.fillna(0.0)

# **MODEL PREPARATION**

In [ ]:
cont_features =[col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_"}]
cat_features = []
target = train["Y1_mean"]

# Description
* To deal with categorical features i made a pipeline containing leave one out encoder and label encoder however i didnt felt the need of using it since all the features i used were continuous except category1 and category2.I used label encoder before grouping the data on the categorical features.

In [ ]:
# from category_encoders import LeaveOneOutEncoder
# from sklearn.preprocessing import LabelEncoder

xgb_cat_features = []
lgb_cat_features = []
cb_cat_features = []
ridge_cat_features = []
sgd_cat_features = []
hgbc_cat_features = []

# loo_features = []
# le_features = []
# def loo_encode(train_df, test_df, column):
#     loo = LeaveOneOutEncoder()
#     new_feature = "{}_loo".format(column)
#     loo.fit(train[column], target)
#     train[column] = loo.transform(train[column])
#     test[column] = loo.transform(test[column])
#     return new_feature

# for feature in cat_features:
#     loo_features.append(loo_encode(train, test, feature))
# #     le_features.append(label_encode(train, test, feature))
    
# # xgb_cat_features.extend(loo_features)
# # lgb_cat_features.extend(le_features)
# cb_cat_features.extend(cat_features)
# # ridge_cat_features.extend(loo_features)
# # sgd_cat_features.extend(loo_features)
# # hgbc_cat_features.extend(loo_features)

# MODELS DESCRIPTION
* four models were used and there output was averaged
>      1.   lightgbm regressor
>      2.   catboost regressor
>      3.   xgboost regressor
>      4.   hyper gradient boosting regressor
* one pipeline was for predicting Y1 and other for prediction Y2


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
# from sklearn.calibration import CalibratedRegressorCV

from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


def rmse(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred))))
def feval_rmse(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSE', rmse(y_true, y_pred), False
random_state = 2021
n_folds = 10
k_fold = KFold(n_splits=n_folds, random_state=random_state, shuffle=True)

xgb_train_preds = np.zeros(len(train.index), )
xgb_test_preds = np.zeros(len(test.index), )
xgb_features = xgb_cat_features + cont_features

lgb_train_preds = np.zeros(len(train.index), )
lgb_test_preds = np.zeros(len(test.index), )
lgb_features = lgb_cat_features + cont_features

cb_train_preds = np.zeros(len(train.index), )
cb_test_preds = np.zeros(len(test.index), )
cb_features = cb_cat_features + cont_features

# ridge_train_preds = np.zeros(len(train.index), )
# ridge_test_preds = np.zeros(len(test.index), )
# ridge_features = ridge_cat_features + cont_features

# sgd_train_preds = np.zeros(len(train.index), )
# sgd_test_preds = np.zeros(len(test.index), )
# sgd_features = sgd_cat_features + cont_features

hgbc_train_preds = np.zeros(len(train.index), )
hgbc_test_preds = np.zeros(len(test.index), )
hgbc_features = hgbc_cat_features + cont_features

for fold, (train_index, test_index) in enumerate(k_fold.split(train)):
    print("--> Fold {}".format(fold + 1))
    y_train = target.iloc[train_index]
    y_valid = target.iloc[test_index]

    xgb_x_train = pandas.DataFrame(train[xgb_features].iloc[train_index])
    xgb_x_valid = pandas.DataFrame(train[xgb_features].iloc[test_index])

    lgb_x_train = pandas.DataFrame(train[lgb_features].iloc[train_index])
    lgb_x_valid = pandas.DataFrame(train[lgb_features].iloc[test_index])

    cb_x_train = pandas.DataFrame(train[cb_features].iloc[train_index])
    cb_x_valid = pandas.DataFrame(train[cb_features].iloc[test_index])

#     ridge_x_train = pandas.DataFrame(train[ridge_features].iloc[train_index])
#     ridge_x_valid = pandas.DataFrame(train[ridge_features].iloc[test_index])

#     sgd_x_train = pandas.DataFrame(train[sgd_features].iloc[train_index])
#     sgd_x_valid = pandas.DataFrame(train[sgd_features].iloc[test_index])

    hgbc_x_train = pandas.DataFrame(train[hgbc_features].iloc[train_index])
    hgbc_x_valid = pandas.DataFrame(train[hgbc_features].iloc[test_index])

    xgb_model = XGBRegressor(
        seed=2021,
        n_estimators=5000,
        verbosity=1,
        eval_metric="rmse",
        tree_method="gpu_hist",
        gpu_id=0,
        alpha=7.105038963844129,
        colsample_bytree=0.25505629740052566,
        gamma=0.4999381950212869,
        reg_lambda=1.7256912198205319,
        learning_rate=0.011823142071967673,
        max_bin=338,
        max_depth=8,
        min_child_weight=2.286836198630466,
        subsample=0.618417952155855,
    )
    xgb_model.fit(
        xgb_x_train,
        y_train,
        eval_set=[(xgb_x_valid, y_valid)], 
        verbose=200,
        early_stopping_rounds=500
    )

    train_oof_preds = xgb_model.predict(xgb_x_valid)
    test_oof_preds = xgb_model.predict(test[xgb_features])
    xgb_train_preds[test_index] = train_oof_preds
    xgb_test_preds += test_oof_preds / n_folds
    print(": XGB - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    seed0=2021
    params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
    train_dataset = lgb.Dataset(lgb_x_train, y_train)
    val_dataset = lgb.Dataset(lgb_x_valid, y_valid)
    model = lgb.train(params = params0,
                          num_boost_round=5000,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=1000,
                          feval = feval_rmse)
    
#     lgb.fit(
#         lgb_x_train,
#         y_train,
#         eval_set=[(lgb_x_valid, y_valid)], 
#         verbose=200,
#     )
    train_oof_preds = model.predict(lgb_x_valid)
    test_oof_preds = model.predict(test[lgb_features])
    lgb_train_preds[test_index] = train_oof_preds
    lgb_test_preds += test_oof_preds / n_folds
    print(": LGB- RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))

    cb_model = CatBoostRegressor(
        verbose=0,
        eval_metric="RMSE",
        loss_function="RMSE",
        random_state=random_state,
        num_boost_round=5000,
        od_type="Iter",
        od_wait=1000,
        task_type="GPU",
        devices="0",
        cat_features=[x for x in range(len(cb_cat_features))],
        bagging_temperature=1.288692494969795,
        grow_policy="Depthwise",
        l2_leaf_reg=9.847870133539244,
        learning_rate=0.01877982653902465,
        max_depth=8,
        min_data_in_leaf=1,
        penalties_coefficient=2.1176668909602734,
    )
    cb_model.fit(
        cb_x_train,
        y_train,
        eval_set=[(cb_x_valid, y_valid)], 
        verbose=100,
    )

    train_oof_preds = cb_model.predict(cb_x_valid)
    test_oof_preds = cb_model.predict(test[cb_features])
    cb_train_preds[test_index] = train_oof_preds
    cb_test_preds += test_oof_preds / n_folds
    print(": CATBOOST - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    
#     ridge_model = CalibratedClassifierCV(
#         RidgeClassifier(random_state=random_state),
#         cv=3,
#     )
#     ridge_model.fit(
#         ridge_x_train,
#         y_train,
#     )

#     train_oof_preds = ridge_model.predict(ridge_x_valid)[:,-1]
#     test_oof_preds = ridge_model.predict(test[ridge_features])
#     ridge_train_preds[test_index] = train_oof_preds
#     ridge_test_preds += test_oof_preds / n_folds
#     print(": Ridge - ROC AUC Score = {}".format(roc_auc_score(y_valid, train_oof_preds, average="micro")))
    
#     sgd_model = CalibratedClassifierCV(
#         SGDClassifier(
#             random_state=random_state,
#             n_jobs=-1,
#             loss="squared_hinge",
#         ),
#         cv=3,
#     )
#     sgd_model.fit(
#         sgd_x_train,
#         y_train,
#     )

#     train_oof_preds = sgd_model.predict(sgd_x_valid)[:,-1]
#     test_oof_preds = sgd_model.predict(test[sgd_features])[:,-1]
#     sgd_train_preds[test_index] = train_oof_preds
#     sgd_test_preds += test_oof_preds / n_folds
#     print(": SGD - ROC AUC Score = {}".format(roc_auc_score(y_valid, train_oof_preds, average="micro")))
    
    hgbc_model = HistGradientBoostingRegressor(
        l2_regularization=1.766059063693552,
        learning_rate=0.10675193678150449,
        max_bins=128,
        max_depth=31,
        max_leaf_nodes=185,
        random_state=2021
    )
    hgbc_model.fit(
        hgbc_x_train,
        y_train,
    )

    train_oof_preds = hgbc_model.predict(hgbc_x_valid)
    test_oof_preds = hgbc_model.predict(test[hgbc_features])
    hgbc_train_preds[test_index] = train_oof_preds
    hgbc_test_preds += test_oof_preds / n_folds
    print(": HGBC- RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    
    
print("--> Overall metrics")
print(": XGB - rmse = {}".format(rmse(target, xgb_train_preds)))
print(": LGB - rmse = {}".format(rmse(target, lgb_train_preds)))
print(": CB - rmse = {}".format(rmse(target, cb_train_preds)))
print(": HGBC - rmse= {}".format(rmse(target, hgbc_train_preds)))



In [ ]:
from scipy.special import expit
from sklearn.calibration import CalibratedClassifierCV

random_state = 2021
n_folds = 10
# k_fold =KFold(n_splits=n_folds, random_state=random_state, shuffle=True)

l1_train = pandas.DataFrame(data={
    "xgb": xgb_train_preds.tolist(),
    "lgb": lgb_train_preds.tolist(),
    "cb": cb_train_preds.tolist(),
#     "ridge": ridge_train_preds.tolist(),
#     "sgd": sgd_train_preds.tolist(),
    "hgbc": hgbc_train_preds.tolist(),
    "target": target.tolist()
})
l1_test = pandas.DataFrame(data={
    "xgb": xgb_test_preds.tolist(),
    "lgb": lgb_test_preds.tolist(),
    "cb": cb_test_preds.tolist(),
#     "sgd": sgd_test_preds.tolist(),
#     "ridge": ridge_test_preds.tolist(),    
    "hgbc": hgbc_test_preds.tolist(),
})

train_preds = np.zeros(len(l1_train.index), )
test_preds = np.zeros(len(l1_test.index), )
features = ["xgb", "lgb","cb", "hgbc"]
x_train = pandas.DataFrame(l1_test[features])
test_preds=(x_train["xgb"]+x_train["lgb"]+x_train["cb"]+x_train["hgbc"])/4

In [ ]:
submission = pandas.read_csv("../input/beyond-analysis/sample_submission_random.csv")
submission["Y1"] = test_preds.tolist()
submission

In [ ]:

target=train["Y2_mean"]
import warnings
warnings.filterwarnings("ignore")
import pandas
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import Ridge
from sklearn.linear_model import SGDRegressor
# from sklearn.calibration import CalibratedRegressorCV

from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingRegressor

from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score


def rmse(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred))))
def feval_rmse(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSE', rmse(y_true, y_pred), False
random_state = 2021
n_folds = 10
k_fold = KFold(n_splits=n_folds, random_state=random_state, shuffle=True)

xgb_train_preds = np.zeros(len(train.index), )
xgb_test_preds = np.zeros(len(test.index), )
xgb_features = xgb_cat_features + cont_features

lgb_train_preds = np.zeros(len(train.index), )
lgb_test_preds = np.zeros(len(test.index), )
lgb_features = lgb_cat_features + cont_features

cb_train_preds = np.zeros(len(train.index), )
cb_test_preds = np.zeros(len(test.index), )
cb_features = cb_cat_features + cont_features

ridge_train_preds = np.zeros(len(train.index), )
ridge_test_preds = np.zeros(len(test.index), )
ridge_features = ridge_cat_features + cont_features

sgd_train_preds = np.zeros(len(train.index), )
sgd_test_preds = np.zeros(len(test.index), )
sgd_features = sgd_cat_features + cont_features

hgbc_train_preds = np.zeros(len(train.index), )
hgbc_test_preds = np.zeros(len(test.index), )
hgbc_features = hgbc_cat_features + cont_features

for fold, (train_index, test_index) in enumerate(k_fold.split(train)):
    print("--> Fold {}".format(fold + 1))
    y_train = target.iloc[train_index]
    y_valid = target.iloc[test_index]

    xgb_x_train = pandas.DataFrame(train[xgb_features].iloc[train_index])
    xgb_x_valid = pandas.DataFrame(train[xgb_features].iloc[test_index])

    lgb_x_train = pandas.DataFrame(train[lgb_features].iloc[train_index])
    lgb_x_valid = pandas.DataFrame(train[lgb_features].iloc[test_index])

    cb_x_train = pandas.DataFrame(train[cb_features].iloc[train_index])
    cb_x_valid = pandas.DataFrame(train[cb_features].iloc[test_index])

#     ridge_x_train = pandas.DataFrame(train[ridge_features].iloc[train_index])
#     ridge_x_valid = pandas.DataFrame(train[ridge_features].iloc[test_index])

#     sgd_x_train = pandas.DataFrame(train[sgd_features].iloc[train_index])
#     sgd_x_valid = pandas.DataFrame(train[sgd_features].iloc[test_index])

    hgbc_x_train = pandas.DataFrame(train[hgbc_features].iloc[train_index])
    hgbc_x_valid = pandas.DataFrame(train[hgbc_features].iloc[test_index])

    xgb_model = XGBRegressor(
        seed=2021,
        n_estimators=4000,
        verbosity=1,
        eval_metric="rmse",
        tree_method="gpu_hist",
        gpu_id=0,
        alpha=7.105038963844129,
        colsample_bytree=0.25505629740052566,
        gamma=0.4999381950212869,
        reg_lambda=1.7256912198205319,
        learning_rate=0.011823142071967673,
        max_bin=338,
        max_depth=8,
        min_child_weight=2.286836198630466,
        subsample=0.618417952155855,
    )
    xgb_model.fit(
        xgb_x_train,
        y_train,
        eval_set=[(xgb_x_valid, y_valid)], 
        verbose=200,
        early_stopping_rounds=500
    )

    train_oof_preds = xgb_model.predict(xgb_x_valid)
    test_oof_preds = xgb_model.predict(test[xgb_features])
    xgb_train_preds[test_index] = train_oof_preds
    xgb_test_preds += test_oof_preds / n_folds
    print(": XGB - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    seed0=2021
    params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
    train_dataset = lgb.Dataset(lgb_x_train, y_train)
    val_dataset = lgb.Dataset(lgb_x_valid, y_valid)
    model = lgb.train(params = params0,
                          num_boost_round=5000,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=1000,
                          feval = feval_rmse)
    
#     lgb.fit(
#         lgb_x_train,
#         y_train,
#         eval_set=[(lgb_x_valid, y_valid)], 
#         verbose=200,
#     )
    train_oof_preds = model.predict(lgb_x_valid)
    test_oof_preds = model.predict(test[lgb_features])
    lgb_train_preds[test_index] = train_oof_preds
    lgb_test_preds += test_oof_preds / n_folds
    print(": lgb - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))

    cb_model = CatBoostRegressor(
        verbose=0,
        eval_metric="RMSE",
        loss_function="RMSE",
        random_state=random_state,
        num_boost_round=5000,
        od_type="Iter",
        od_wait=500,
        task_type="GPU",
        devices="0",
        cat_features=[x for x in range(len(cb_cat_features))],
        bagging_temperature=1.288692494969795,
        grow_policy="Depthwise",
        l2_leaf_reg=9.847870133539244,
        learning_rate=0.01877982653902465,
        max_depth=8,
        min_data_in_leaf=1,
        penalties_coefficient=2.1176668909602734,
    )
    cb_model.fit(
        cb_x_train,
        y_train,
        eval_set=[(cb_x_valid, y_valid)], 
        verbose=100,
    )

    train_oof_preds = cb_model.predict(cb_x_valid)
    test_oof_preds = cb_model.predict(test[cb_features])
    cb_train_preds[test_index] = train_oof_preds
    cb_test_preds += test_oof_preds / n_folds
    print(": CATBOOST - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    
#     ridge_model = CalibratedClassifierCV(
#         RidgeClassifier(random_state=random_state),
#         cv=3,
#     )
#     ridge_model.fit(
#         ridge_x_train,
#         y_train,
#     )

#     train_oof_preds = ridge_model.predict(ridge_x_valid)[:,-1]
#     test_oof_preds = ridge_model.predict(test[ridge_features])
#     ridge_train_preds[test_index] = train_oof_preds
#     ridge_test_preds += test_oof_preds / n_folds
#     print(": Ridge - ROC AUC Score = {}".format(roc_auc_score(y_valid, train_oof_preds, average="micro")))
    
#     sgd_model = CalibratedClassifierCV(
#         SGDClassifier(
#             random_state=random_state,
#             n_jobs=-1,
#             loss="squared_hinge",
#         ),
#         cv=3,
#     )
#     sgd_model.fit(
#         sgd_x_train,
#         y_train,
#     )

#     train_oof_preds = sgd_model.predict(sgd_x_valid)[:,-1]
#     test_oof_preds = sgd_model.predict(test[sgd_features])[:,-1]
#     sgd_train_preds[test_index] = train_oof_preds
#     sgd_test_preds += test_oof_preds / n_folds
#     print(": SGD - ROC AUC Score = {}".format(roc_auc_score(y_valid, train_oof_preds, average="micro")))
    
    hgbc_model = HistGradientBoostingRegressor(
        l2_regularization=1.766059063693552,
        learning_rate=0.10675193678150449,
        max_bins=128,
        max_depth=31,
        max_leaf_nodes=185,
        random_state=2021
    )
    hgbc_model.fit(
        hgbc_x_train,
        y_train,
    )

    train_oof_preds = hgbc_model.predict(hgbc_x_valid)
    test_oof_preds = hgbc_model.predict(test[hgbc_features])
    hgbc_train_preds[test_index] = train_oof_preds
    hgbc_test_preds += test_oof_preds / n_folds
    print(": hgbc - RMSE Score = {}".format(rmse(y_valid, train_oof_preds)))
    
    
print("--> Overall metrics")
print(": XGB - rmse = {}".format(rmse(target, xgb_train_preds)))
print(": LGB - rmse = {}".format(rmse(target, lgb_train_preds)))
print(": CB - rmse = {}".format(rmse(target, cb_train_preds)))
# print(": Ridge - ROC AUC Score = {}".format(roc_auc_score(target, ridge_train_preds, average="micro")))
# print(": SGD - ROC AUC Score = {}".format(roc_auc_score(target, sgd_train_preds, average="micro")))
print(": HGBC - rmse= {}".format(rmse(target, hgbc_train_preds)))


In [ ]:

from scipy.special import expit
from sklearn.calibration import CalibratedClassifierCV

random_state = 2021
n_folds = 10
# k_fold =KFold(n_splits=n_folds, random_state=random_state, shuffle=True)

l1_train = pandas.DataFrame(data={
    "xgb": xgb_train_preds.tolist(),
    "lgb": lgb_train_preds.tolist(),
    "cb": cb_train_preds.tolist(),
#     "ridge": ridge_train_preds.tolist(),
#     "sgd": sgd_train_preds.tolist(),
    "hgbc": hgbc_train_preds.tolist(),
    "target": target.tolist()
})
l1_test = pandas.DataFrame(data={
    "xgb": xgb_test_preds.tolist(),
    "lgb": lgb_test_preds.tolist(),
    "cb": cb_test_preds.tolist(),
#     "sgd": sgd_test_preds.tolist(),
#     "ridge": ridge_test_preds.tolist(),    
    "hgbc": hgbc_test_preds.tolist(),
})

train_preds = np.zeros(len(l1_train.index), )
test_preds1 = np.zeros(len(l1_test.index), )
features = ["xgb", "lgb","cb", "hgbc"]
x_train = pandas.DataFrame(l1_test[features])
test_preds1=(x_train["xgb"]+x_train["lgb"]+x_train["cb"]+x_train["hgbc"])/4


In [ ]:
submission["Y2"] = test_preds1.tolist()
submission
submission.to_csv("submission.csv", index=False)

In [ ]:
from tensorflow.keras.layers import Input,Dense,Dropout
from tensorflow.keras import Model
from  tensorflow.keras.regularizers import l2
import tensorflow as tf
from numpy.random import seed
seed(42)
import tensorflow as tf
tf.random.set_seed(42)
from tensorflow import keras
import numpy as np
from keras import backend as K


def root_mean_squared_error(y_true, y_pred):
         return K.sqrt(K.mean(K.square( (y_true - y_pred) )))
    
    
from keras.backend import sigmoid
def swish(x, beta = 1):
    return (x * sigmoid(beta * x))
from keras.utils.generic_utils import get_custom_objects
from keras.layers import Activation
get_custom_objects().update({'swish': Activation(swish)})

In [ ]:
features = [col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_","SEQUENCE_NO_size"}]

# **DAE is Used to add diversity to the model**

In [ ]:
def get_DAE():
    # denoising autoencoder
    inputs = Input((266,))
    x = Dense(500, activation='swish')(inputs) # 1500 original
    x = Dense(500, activation='swish', name="feature")(x) # 1500 original
    x = Dense(500, activation='swish')(x) # 1500 original
    outputs = Dense(266, activation='linear')(x)
    model = Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer='adam', loss=root_mean_squared_error)
    return model

In [ ]:
alldata = pd.concat([train[features],test[features]],axis=0)
print(alldata.shape)
autoencoder = get_DAE()
autoencoder.fit(alldata[features], alldata[features],
                    epochs=50,
                    batch_size=256,
                    shuffle=True
                    )

In [ ]:
test_denoised = test.copy()
test[features] = autoencoder.predict(test_denoised[features])
train_denoised = train.copy()
train[features] = autoencoder.predict(train_denoised[features])

In [ ]:


from sklearn.model_selection import KFold
folds=[]
kfold = KFold(n_splits = 5, random_state = 2021, shuffle = True)
for fold, (trn_ind, val_ind) in enumerate(kfold.split(train)):
    print(trn_ind,val_ind)
    folds.append([trn_ind,val_ind])



In [ ]:
target=['Y1_mean','Y2_mean']

In [ ]:
# from sklearn.model_selection import KFold
import lightgbm as lgb

seed0=2021
params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
seed1=42
params1 = {
        'learning_rate': 0.1,        
        'lambda_l1': 2,
        'lambda_l2': 7,
        'num_leaves': 800,
        'min_sum_hessian_in_leaf': 20,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.9,
        'bagging_freq': 42,
        'min_data_in_leaf': 700,
        'max_depth': 4,
        'categorical_column':[0],
        'seed': seed1,
        'feature_fraction_seed': seed1,
        'bagging_seed': seed1,
        'drop_seed': seed1,
        'data_random_seed': seed1,
        'objective': 'rmse',
        'boosting': 'gbdt',
        'verbosity': -1,
        'n_jobs':-1,
    }
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred))))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False
# X=train[features]

def train_and_evaluate_lgb(train, test, params, features):
    # Hyperparammeters (just basic)
    y = train[target[0]]
    X=train[features]
    # Create out of folds array
    oof_predictions = np.zeros(train.shape[0])
    # Create test array to store predictions
    test_predictions = np.zeros(test.shape[0])
    # Create a KFold object
    kfold = KFold(n_splits = 5, random_state = 2021, shuffle = True)
    # Iterate through each fold
    fold=0
    for trn_ind,val_ind in folds:
        print(f'Training fold {fold + 1}')
        fold+=1
        x_train, x_val = X.iloc[trn_ind], X.iloc[val_ind]
        y_train, y_val = y.iloc[trn_ind], y.iloc[val_ind]
        # Root mean squared percentage error weights
        train_dataset = lgb.Dataset(x_train[features], y_train)
        val_dataset = lgb.Dataset(x_val[features], y_val)
        model = lgb.train(params = params,
                          num_boost_round=2000,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=250,
                          feval = feval_rmspe)
        # Add predictions to the out of folds array
        oof_predictions[val_ind] = model.predict(x_val[features])
        # Predict the test set
        test_predictions += model.predict(test[features]) / 5
    rmspe_score = rmspe(y, oof_predictions)
    print(f'Our out of folds RMSPE is {rmspe_score}')
    lgb.plot_importance(model,max_num_features=20)
    # Return test predictions
    return test_predictions
# Traing and evaluate
features = [col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_"}]
#features = [col for col in features if col not in cols_drop]
predictions_lgb= train_and_evaluate_lgb(train_denoised, test_denoised, params0, features)
test_denoised[target[0]] = predictions_lgb

# cont_features =[col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_","STATUS_CHECK_mean","SEQUENCE_NO_size"}]
# cat_features = ["STATUS_CHECK_mean","SEQUENCE_NO_size"]

In [ ]:
from sklearn.model_selection import KFold
import lightgbm as lgb

seed0=2021
params0 = {
    'objective': 'rmse',
    'boosting_type': 'gbdt',
    'max_depth': -1,
    'max_bin':100,
    'min_data_in_leaf':500,
    'learning_rate': 0.05,
    'subsample': 0.72,
    'subsample_freq': 4,
    'feature_fraction': 0.5,
    'lambda_l1': 0.5,
    'lambda_l2': 1.0,
    'categorical_column':[0],
    'seed':seed0,
    'feature_fraction_seed': seed0,
    'bagging_seed': seed0,
    'drop_seed': seed0,
    'data_random_seed': seed0,
    'n_jobs':-1,
    'verbose': -1}
seed1=42
params1 = {
        'learning_rate': 0.1,        
        'lambda_l1': 2,
        'lambda_l2': 7,
        'num_leaves': 800,
        'min_sum_hessian_in_leaf': 20,
        'feature_fraction': 0.8,
        'feature_fraction_bynode': 0.8,
        'bagging_fraction': 0.9,
        'bagging_freq': 42,
        'min_data_in_leaf': 700,
        'max_depth': 4,
        'categorical_column':[0],
        'seed': seed1,
        'feature_fraction_seed': seed1,
        'bagging_seed': seed1,
        'drop_seed': seed1,
        'data_random_seed': seed1,
        'objective': 'rmse',
        'boosting': 'gbdt',
        'verbosity': -1,
        'n_jobs':-1,
    }
# Function to early stop with root mean squared percentage error
def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(np.square((y_true - y_pred))))

def feval_rmspe(y_pred, lgb_train):
    y_true = lgb_train.get_label()
    return 'RMSPE', rmspe(y_true, y_pred), False
X=train_denoised[features]

def train_and_evaluate_lgb(train, test, params, features):
    # Hyperparammeters (just basic)
    y = train[target[1]]
    X=train[features]
    # Create out of folds array
    oof_predictions = np.zeros(train.shape[0])
    # Create test array to store predictions
    test_predictions = np.zeros(test.shape[0])
    # Create a KFold object
    kfold = KFold(n_splits = 5, random_state = 2021, shuffle = True)
    # Iterate through each fold
    fold=0
    for trn_ind,val_ind in folds:
        print(f'Training fold {fold + 1}')
        fold+=1
        x_train, x_val = X.iloc[trn_ind], X.iloc[val_ind]
        y_train, y_val = y.iloc[trn_ind], y.iloc[val_ind]
        # Root mean squared percentage error weights
        train_dataset = lgb.Dataset(x_train[features], y_train)
        val_dataset = lgb.Dataset(x_val[features], y_val)
        model = lgb.train(params = params,
                          num_boost_round=2000,
                          train_set = train_dataset, 
                          valid_sets = [train_dataset, val_dataset], 
                          verbose_eval = 250,
                          early_stopping_rounds=500,
                          feval = feval_rmspe)
        # Add predictions to the out of folds array
        oof_predictions[val_ind] = model.predict(x_val[features])
        # Predict the test set
        test_predictions += model.predict(test[features]) / 5
    rmspe_score = rmspe(y, oof_predictions)
    print(f'Our out of folds RMSPE is {rmspe_score}')
    lgb.plot_importance(model,max_num_features=20)
    # Return test predictions
    return test_predictions
# Traing and evaluate
features = [col for col in train.columns if col not in {"Y1_mean","Y2_mean","UNIQUE_IDENTIFIER_"}]
#features = [col for col in features if col not in cols_drop]
predictions_lgb= train_and_evaluate_lgb(train_denoised, test_denoised, params0, features)
test_denoised[target[1]] = predictions_lgb

In [ ]:
submission1 = pandas.read_csv("../input/beyond-analysis/sample_submission_random.csv")
submission1['Y1']=0.50*submission['Y1']+0.50*test_denoised['Y1_mean']
submission1['Y2']=0.50*submission['Y2']+0.50*test_denoised['Y2_mean']

In [ ]:
submission1.to_csv("submission1.csv", index=False)

# **CV OBSERVATIONS**
*  cv was correlating with public leaderboard.
* i was able to get best rmse for y1 round of  6.70 and for y2 =119.31